# nb05 — `pass^k`: confiabilidade, e o que a média esconde (T29)

**O que este notebook faz:** lê os scores N1/N2 da bateria principal (288 execuções: 18 cenários
de test × 2 modelos × 8 `sample_seed`) e produz as duas figuras da T29:

1. **as curvas de `pass^k`, k = 1..8, contra a média simples** —
   `figures/fig09_passk_curvas.png`, onde a ordem entre os modelos se inverte;
2. **a decomposição da variância** — `figures/fig10_decomposicao_variancia.png`, que mostra
   *por que* ela se inverte.

> **A aritmética não mora aqui.** O estimador é `tapieval.scoring.passk` (14 testes) e a
> agregação é `tapieval.scoring.estabilidade` (18 testes). Um notebook que reimplementasse a
> conta produziria uma segunda versão dela, e a que aparece na figura seria justamente a que
> ninguém testou — mesma regra do nb03 e do nb04.

---

## ⚠️ Metade da T29 não tem dado, e não é por análise que se resolve

A T29 pedia `pass^k` **em ambiente fixo e livre**, com a área entre as curvas sombreada — essa
área *é* a H4 (`METRICAS §7.2`). Ela não está aqui, e a razão está fechada desde 30/08:

- **a bateria de ambiente foi cortada** (A16), junto com a metamórfica, para a noite de GPU
  fechar em 11,42 h;
- **e ela não roda hoje de qualquer modo**: `runner/matriz.py` não tem eixo de `env_seed` — a
  seed do ambiente entra na célula como constante lida do YAML do cenário. O bloqueio e a
  extensão mínima que o destrava estão escritos em `configs/bateria_ambiente.yaml`.

Nas três baterias no disco, **`env_seed` é constante por célula** — `tests/test_estabilidade.py::
test_a_bateria_no_disco_nao_tem_eixo_de_ambiente` confere isso célula a célula e é o tripwire:
no dia em que o eixo existir, ele falha e manda reabrir H4.

**Não há substituto por reamostragem.** Reamostrar as 8 seeds do *mesmo* mundo mede a variância
do modelo — que é exatamente o braço que já se tem. O que falta é o outro mundo.

O que a T29 entrega, então, é o braço de ambiente fixo: `pass^k` do modelo, e uma decomposição
de variância que é **entre cenários × dentro do cenário**, não modelo × ambiente.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))

from tapieval import figuras as fg
from tapieval.scoring.bateria import ler_scores
from tapieval.scoring.estabilidade import (
    K_MAXIMO_DA_BATERIA,
    cruzamento,
    curva,
    decomposicao_de_variancia,
    sucesso_da_run,
    vetores_por_cenario,
)

PRINCIPAL = RAIZ / "runs" / "principal_2026_08" / "scores.jsonl"
FIGURAS = RAIZ / "figures"
MODELOS = ("qwen3-8b", "qwen3-14b")
TRATOS = ("incluir", "falha", "excluir")

scores = ler_scores(PRINCIPAL)
len(scores)

288

In [2]:
%load_ext watermark
%watermark -u -d -v -m -p pandas,plotly,kaleido

Last updated: 2026-09-01

Python implementation: CPython
Python version       : 3.14.7
IPython version      : 9.16.1

pandas : 3.0.5
plotly : 6.9.0
kaleido: 1.3.0

Compiler    : Clang 21.0.0 (clang-2100.1.1.101)
OS          : Darwin
Release     : 25.5.0
Machine     : arm64
Processor   : arm
CPU cores   : 10
Architecture: 64bit



---

## 1. A métrica oficial é a reta zero

`sucesso_binario` de `METRICAS §6.5` — nenhuma falha S0, S1 ou S2 — reprova **as 288 execuções**.
`pass^k` sobre ela não decai, não ordena os modelos e não tem o que esconder: é o X33 medido
onde ele mais dói. Por isso tudo daqui para baixo usa a lente `sem_s2`, a mitigação que o X33
propôs — que tem um defeito próprio, e é o assunto da seção 2.

In [3]:
lente_oficial = pd.DataFrame(
    [
        {
            "modelo": m,
            "aprovações (nominal)": sum(sucesso_da_run(s, "nominal") for s in scores if s.model_key == m),
            "aprovações (sem S2)": sum(sucesso_da_run(s, "sem_s2") for s in scores if s.model_key == m),
            "execuções": sum(s.model_key == m for s in scores),
        }
        for m in MODELOS
    ]
).set_index("modelo")
lente_oficial

,aprovações (nominal),aprovações (sem S2),execuções
modelo,,,
qwen3-8b,0,38,144
qwen3-14b,0,55,144


---

## 2. As 37 execuções sem decisão, e o que elas fazem com a manchete

37 execuções têm `pontuavel = False`, todas por `decisao_prevista is None` — o trace não tem
`DecisionEvent` nem ato observável. **30 são do 14B**: é o X31, a taxa de `parse_erro` 15× maior
do modelo maior.

O classificador dá a essas execuções **só códigos de processo** (P1, P2, P3, P5, P6) e nenhum é
S0 ou S1 — então **todas as 37 passam na lente `sem_s2`**. A execução em que o agente não decidiu
nada é contada como sucesso pela lente proposta para consertar o corte, e 30 desses sucessos são
de um modelo só.

`TratoDeIndecisa` põe as três leituras na mesa em vez de escolher uma calada. O padrão do módulo
é `excluir` — a regra já congelada em `passk.py` —, mas **a premissa dela não vale para este
motivo**: lá o argumento é que converter para `False` transformaria defeito do *instrumento* em
erro do *modelo*, e o X31 concluiu o contrário (*"não é ruído, e não é a máquina: é o modelo"*).
Para `decisao_prevista is None`, `falha` é a leitura defensável — e é a única das três que chega
a k = 8 sem creditar o X31 como acerto.

In [4]:
# Toda execução sem decisão passa na lente `sem_s2`, então ela entra INTEIRA nas aprovações
# do modelo. A coluna que importa é a última: quanto da "aprovação" é isso.
composicao = pd.DataFrame(
    [
        {
            "modelo": m,
            "aprovações em sem_s2": (
                aprov := sum(sucesso_da_run(s, "sem_s2") for s in scores if s.model_key == m)
            ),
            "delas, sem decisão": (
                indecisas := sum(
                    1 for s in scores if s.model_key == m and not s.pontuavel
                )
            ),
            "fração da aprovação que é o X31": indecisas / aprov,
        }
        for m in MODELOS
    ]
).set_index("modelo")
composicao.style.format({"fração da aprovação que é o X31": "{:.0%}"})

,aprovações em sem_s2,"delas, sem decisão",fração da aprovação que é o X31
modelo,,,
qwen3-8b,38,7,18%
qwen3-14b,55,30,55%


---

## 3. A curva — e a inversão

`figures/fig09_passk_curvas.png`.

**O que ela mostra:** a média simples põe o 14B na frente (38,2% contra 26,4% na leitura
`incluir`), e **`pass^k` inverte a ordem a partir de k = 3**. Nas outras duas leituras o 8B já
começa à frente. Ou seja: a inversão não depende de como se tratam as 37 — o que depende é
apenas *em que k* ela acontece.

**O que ela NÃO mostra:** nada sobre ambiente (ver o cabeçalho). E o `pass^8 = 0,000` das duas
curvas não é artefato do corte — **nenhum cenário é entregue nas 8 seeds por nenhum dos dois
modelos**, e é isso que impede de ler a inversão como *"então use o 8B"*.

In [5]:
vet = {
    (m, t): vetores_por_cenario(scores, model_key=m, lente="sem_s2", trato=t)
    for m in MODELOS
    for t in TRATOS
}
cur = {ch: curva(v) for ch, v in vet.items()}

TINTA, TINTA2, SUPERFICIE = fg.TINTA, fg.TINTA2, fg.SUPERFICIE
AZUL, AMBAR, VERDE, CINZA = fg.AZUL, fg.AMBAR, fg.VERDE, fg.CINZA
COR = {"qwen3-8b": AZUL, "qwen3-14b": AMBAR}
ROTULO = {"qwen3-8b": "8B", "qwen3-14b": "14B"}
TITULO_DO_TRATO = {
    "incluir": "<b>incluir</b><br><sub>as 37 sem decisão contam como sucesso</sub>",
    "falha": "<b>falha</b><br><sub>não entregar é não passar</sub>",
    "excluir": "<b>excluir</b><br><sub>fora do vetor — a curva trunca</sub>",
}

pd.DataFrame(
    [
        {
            "modelo": ROTULO[m],
            "trato": t,
            "média simples": cur[(m, t)].media_simples,
            "pass^1": cur[(m, t)].passk[1],
            "pass^4": cur[(m, t)].passk[4],
            "pass^8": cur[(m, t)].passk[8],
            "k máx. estimável": cur[(m, t)].k_maximo_estimavel,
            "descartadas": cur[(m, t)].n_descartados,
        }
        for m in MODELOS
        for t in TRATOS
    ]
).set_index(["modelo", "trato"])

média simples    pass^1    pass^4  pass^8  k máx. estimável  \
modelo trato                                                                  
8B     incluir       0.263889  0.263889  0.065079     0.0                 8   
       falha         0.215278  0.215278  0.034127     0.0                 8   
       excluir       0.226277  0.238889  0.045238     NaN                 5   
14B    incluir       0.381944  0.381944  0.046032     0.0                 8   
       falha         0.173611  0.173611  0.007937     0.0                 8   
       excluir       0.219298  0.212169  0.026455     NaN                 4   

                descartadas  
modelo trato                 
8B     incluir            0  
       falha              0  
       excluir            7  
14B    incluir            0  
       falha              0  
       excluir           30

In [6]:
ks = list(range(1, K_MAXIMO_DA_BATERIA + 1))

fig = make_subplots(
    rows=1,
    cols=3,
    shared_yaxes=True,
    horizontal_spacing=0.045,
    subplot_titles=[TITULO_DO_TRATO[t] for t in TRATOS],
)

for col, trato in enumerate(TRATOS, start=1):
    # A lente oficial de §6.5: zero para todo k, nos dois modelos. Fica no fundo.
    fig.add_scatter(
        x=ks, y=[0.0] * len(ks), mode="lines", row=1, col=col,
        line=dict(color=CINZA, width=5), showlegend=False, hoverinfo="skip",
    )
    for modelo in MODELOS:
        c = cur[(modelo, trato)]
        y = [c.passk[k] for k in ks]
        fig.add_scatter(
            x=ks, y=y, mode="lines+markers", row=1, col=col,
            line=dict(color=COR[modelo], width=3),
            marker=dict(size=10, color=COR[modelo]),
            name=ROTULO[modelo], legendgroup=modelo, showlegend=col == 1,
            hovertemplate=f"{ROTULO[modelo]} · pass^%{{x}} = %{{y:.1%}}<extra></extra>",
        )
        # A média simples da MESMA base — a linha que o pass^k contradiz.
        fig.add_scatter(
            x=[0.6, K_MAXIMO_DA_BATERIA + 0.4], y=[c.media_simples] * 2, mode="lines",
            row=1, col=col, line=dict(color=COR[modelo], width=1.5, dash="dot"),
            showlegend=False, hoverinfo="skip",
        )

    k_cruz = cruzamento(cur[("qwen3-8b", trato)], cur[("qwen3-14b", trato)])
    if k_cruz is not None:
        fig.add_vline(
            x=k_cruz, row=1, col=col, line=dict(color=VERDE, width=2, dash="dash"),
        )
        fig.add_annotation(
            row=1, col=col, x=k_cruz + 0.15, y=0.415, xanchor="left", yanchor="middle",
            text=f"<b>8B à frente<br>de k = {k_cruz} em diante</b>",
            font=dict(color=VERDE, size=11), showarrow=False, align="left",
        )

# Cada pontilhada é rotulada na ponta direita, onde nenhuma curva passa.
for modelo in MODELOS:
    fig.add_annotation(
        row=1, col=1, x=8.3, y=cur[(modelo, "incluir")].media_simples,
        xanchor="right", yanchor="bottom", showarrow=False, align="right",
        text=f"<sub>média simples · {ROTULO[modelo]}</sub>",
        font=dict(color=COR[modelo], size=10),
    )
# A lente oficial é anotada ABAIXO da linha do zero, na faixa vazia do eixo.
fig.add_annotation(
    row=1, col=2, x=4.5, y=-0.012, xanchor="center", yanchor="top", showarrow=False,
    text="<sub>lente oficial de §6.5 — <b>zero em todo k, nos dois modelos</b></sub>",
    font=dict(color=TINTA2, size=10), align="center",
)
# Por que a terceira curva acaba antes do fim.
fig.add_annotation(
    row=1, col=3, x=8.3, y=0.30, xanchor="right", yanchor="top", showarrow=False,
    text=("<sub>trunca onde o menor cenário acaba:<br>"
          "5 tentativas (8B), 4 (14B)</sub>"),
    font=dict(color=TINTA2, size=10), align="right",
)

fig.update_layout(
    title=dict(
        text=(
            "<b>A média ordena os modelos; exigir consistência inverte a ordem</b><br>"
            "<sub>pass^k sobre as 8 <i>sample_seed</i> · 18 cenários de test · lente sem S2 · "
            "as três leituras das 37 execuções sem decisão</sub>"
        ),
        font=dict(color=TINTA, size=19), x=0, xanchor="left",
    ),
    plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE, font=dict(color=TINTA2, family=fg.FAMILIA),
    width=1180, height=610, margin=dict(l=80, r=40, t=140, b=105),
    legend=dict(orientation="h", y=-0.20, x=0.5, yanchor="top", xanchor="center"),
)
fig.update_xaxes(
    title=dict(text="k — tentativas que precisam TODAS passar", standoff=12),
    tickmode="array", tickvals=ks, gridcolor=CINZA, zeroline=False,
    range=[0.6, K_MAXIMO_DA_BATERIA + 0.4],
)
fig.update_yaxes(gridcolor=CINZA, zeroline=False, range=[-0.055, 0.46], tickformat=".0%")
fig.update_yaxes(title="pass^k médio entre cenários", row=1, col=1)
for anotacao in fig.layout.annotations[:3]:
    anotacao.font = dict(color=TINTA, size=13)
    anotacao.yshift = -6

fg.exportar(fig, "fig09_passk_curvas", FIGURAS)
print("fig09 ok")

for trato in TRATOS:
    c8, c14 = cur[("qwen3-8b", trato)], cur[("qwen3-14b", trato)]
    print(
        f"{trato:8s} media 8B={c8.media_simples:.3f} 14B={c14.media_simples:.3f} "
        f"cruz={cruzamento(c8, c14)} trunc8B={c8.k_maximo_estimavel} "
        f"trunc14B={c14.k_maximo_estimavel}"
    )
fig.show()

fig09 ok
incluir  media 8B=0.264 14B=0.382 cruz=3 trunc8B=8 trunc14B=8
falha    media 8B=0.215 14B=0.174 cruz=1 trunc8B=8 trunc14B=8
excluir  media 8B=0.226 14B=0.219 cruz=1 trunc8B=5 trunc14B=4


---

## 4. Por que ela inverte — a decomposição

`figures/fig10_decomposicao_variancia.png`.

    Var(Y) = E[Var(Y | cenário)] + Var(E[Y | cenário])
             └── dentro do cenário ──┘   └── entre cenários ──┘

**O que ela mostra:** quase metade da variância do 8B é **entre** cenários (42–49% nos três
tratos) — ele tem cenários que domina e cenários que não. A do 14B é predominantemente **dentro**
do cenário (só 19–35% é entre): ele varia de tentativa para tentativa no mesmo cenário. `pass^k`
cobra exatamente essa segunda; a média simples não vê nenhuma das duas. A direção sobrevive aos
três tratos, ao contrário do nível.

**O que ela NÃO mostra:** **não é a decomposição de H4.** `METRICAS §7.2` chama de decomposição
de variância a separação *modelo × ambiente*, que exige os dois braços. Aqui o ambiente é
**constante** — a pergunta é outra, e `Decomposicao` não tem campo que sugira o contrário.

In [7]:
dec = {ch: decomposicao_de_variancia(v) for ch, v in vet.items()}

fig2 = make_subplots(
    rows=1, cols=2, column_widths=[0.46, 0.54], horizontal_spacing=0.13,
    subplot_titles=[
        "<b>De onde vem a variância</b><br>"
        "<sub>fração do total · nos três tratos</sub>",
        "<b>Por quê: a taxa de sucesso de cada um dos 18 cenários</b><br>"
        "<sub>trato <i>falha</i> · cada ponto é um cenário</sub>",
    ],
)

rotulos_y, fr_entre, fr_dentro, cores = [], [], [], []
for modelo in MODELOS:
    for trato in TRATOS:
        d = dec[(modelo, trato)]
        rotulos_y.append(f"{ROTULO[modelo]} · {trato}")
        fr_entre.append(d.entre / d.total)
        fr_dentro.append(d.dentro / d.total)
        cores.append(COR[modelo])

fig2.add_bar(
    x=fr_entre, y=rotulos_y, orientation="h", row=1, col=1, name="entre cenários (cor do modelo)",
    marker=dict(color=cores), text=[f"{v:.0%}" for v in fr_entre],
    textposition="inside", insidetextanchor="middle",
    textfont=dict(color=SUPERFICIE, size=12),
    hovertemplate="entre cenários · %{x:.1%}<extra></extra>",
)
fig2.add_bar(
    x=fr_dentro, y=rotulos_y, orientation="h", row=1, col=1, name="dentro do cenário",
    marker=dict(color=CINZA), text=[f"{v:.0%}" for v in fr_dentro],
    textposition="inside", insidetextanchor="middle",
    textfont=dict(color=TINTA, size=12),
    hovertemplate="dentro do cenário · %{x:.1%}<extra></extra>",
)

for i, modelo in enumerate(MODELOS):
    taxas = sorted(sum(v) / len(v) for v in vet[(modelo, "falha")].por_cenario.values())
    # Empilhamento centrado: muitos cenários caem na MESMA taxa (várias em zero) e um
    # ponto sobre o outro esconderia a massa que é justamente o achado.
    quantos: dict[float, int] = {}
    for t in taxas:
        quantos[t] = quantos.get(t, 0) + 1
    postos: dict[float, int] = {}
    y = []
    for t in taxas:
        n = postos.get(t, 0)
        postos[t] = n + 1
        y.append(i + (n - (quantos[t] - 1) / 2) * 0.052)
    fig2.add_scatter(
        x=taxas, y=y, mode="markers", row=1, col=2, showlegend=False,
        marker=dict(size=12, color=COR[modelo], opacity=0.8,
                    line=dict(color=SUPERFICIE, width=1.5)),
        hovertemplate=f"{ROTULO[modelo]} · %{{x:.0%}} dos 8 trials<extra></extra>",
    )

# As duas frases descrevem o que os pontos mostram, e o eixo x é o mesmo nos dois.
fig2.add_annotation(
    row=1, col=2, x=1.0, y=1.62, xanchor="right", yanchor="top", showarrow=False,
    text=("<b>comprimido contra o piso</b><br><sub>o melhor cenário do 14B dá 5 de 8;<br>"
          "quase toda a massa fica perto de zero</sub>"),
    font=dict(color=AMBAR, size=11), align="right",
)
fig2.add_annotation(
    row=1, col=2, x=1.0, y=0.62, xanchor="right", yanchor="top", showarrow=False,
    text=("<b>espalhado</b><br><sub>o 8B tem cenário de 7 de 8 — e cenário de 0 de 8:<br>"
          "é essa distância que a coluna azul mede</sub>"),
    font=dict(color=AZUL, size=11), align="right",
)

fig2.update_layout(
    title=dict(
        text=("<b>O 14B erra por inconsistência; o 8B, por dificuldade de cenário</b><br>"
              "<sub>Var(Y) = E[Var(Y | cenário)] + Var(E[Y | cenário]) · ambiente CONSTANTE — "
              "não é a decomposição modelo × ambiente de H4, que não foi rodada</sub>"),
        font=dict(color=TINTA, size=19), x=0, xanchor="left",
    ),
    barmode="stack", plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE,
    font=dict(color=TINTA2, family=fg.FAMILIA), width=1180, height=560,
    margin=dict(l=130, r=45, t=150, b=95),
    legend=dict(orientation="h", y=-0.16, x=0.0, yanchor="top", xanchor="left"),
)
fig2.update_xaxes(row=1, col=1, tickformat=".0%", gridcolor=CINZA, zeroline=False,
                  range=[0, 1], title=dict(text="fração da variância total", standoff=10))
fig2.update_yaxes(row=1, col=1, autorange="reversed", gridcolor=SUPERFICIE)
fig2.update_xaxes(row=1, col=2, tickformat=".0%", gridcolor=CINZA, zeroline=False,
                  range=[-0.06, 1.02],
                  title=dict(text="taxa de sucesso no cenário (de 8 tentativas)", standoff=10))
fig2.update_yaxes(row=1, col=2, tickmode="array", tickvals=[0, 1],
                  ticktext=[f"<b>{ROTULO[m]}</b>" for m in MODELOS],
                  range=[-0.5, 1.95], gridcolor=SUPERFICIE, zeroline=False)
for anotacao in fig2.layout.annotations[:2]:
    anotacao.font = dict(color=TINTA, size=13)

fg.exportar(fig2, "fig10_decomposicao_variancia", FIGURAS)
fig2.show()

---

## 5. Os números para o README e para os slides

Gravados em `docs/anexos/resultados/resultados_passk.json`, e **não** em `docs/anexos/resultados/resultados_h0.json`: aquele arquivo
é reescrito inteiro pelo nb04 a cada execução, e um merge daqui seria apagado na próxima vez que
o nb04 rodasse. Um arquivo por notebook, um dono por arquivo.

In [8]:
resumo = {
    "bateria": "principal_2026_08",
    "n_execucoes": len(scores),
    "n_sem_decisao": sum(1 for s in scores if not s.pontuavel),
    "h4": {
        "calculavel": False,
        "motivo": (
            "env_seed é constante por célula nas três baterias: a bateria de ambiente foi "
            "cortada (A16) e runner/matriz.py não tem eixo de env_seed. Sem os dois braços "
            "não há área entre curvas, e reamostrar o mesmo mundo não substitui o outro."
        ),
    },
    "lente_nominal_aprovacoes": {
        ROTULO[m]: sum(sucesso_da_run(s, "nominal") for s in scores if s.model_key == m)
        for m in MODELOS
    },
    "curvas": {
        f"{ROTULO[m]}|{t}": {
            "media_simples": round(cur[(m, t)].media_simples, 4),
            "passk": {
                str(k): (None if v != v else round(v, 4))
                for k, v in cur[(m, t)].passk.items()
            },
            "k_maximo_estimavel": cur[(m, t)].k_maximo_estimavel,
            "n_trials": cur[(m, t)].n_trials,
            "n_descartados": cur[(m, t)].n_descartados,
        }
        for m in MODELOS
        for t in TRATOS
    },
    "cruzamento_8b_ultrapassa_14b": {
        t: cruzamento(cur[("qwen3-8b", t)], cur[("qwen3-14b", t)]) for t in TRATOS
    },
    "decomposicao": {
        f"{ROTULO[m]}|{t}": {
            "dentro": round(dec[(m, t)].dentro, 4),
            "entre": round(dec[(m, t)].entre, 4),
            "fracao_entre": round(dec[(m, t)].fracao_entre, 4),
        }
        for m in MODELOS
        for t in TRATOS
    },
}
(RAIZ / "docs" / "anexos" / "resultados" / "resultados_passk.json").write_text(
    json.dumps(resumo, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
print(json.dumps(
    {"cruzamento": resumo["cruzamento_8b_ultrapassa_14b"], "decomposicao": resumo["decomposicao"]},
    indent=2, ensure_ascii=False,
))

{
  "cruzamento": {
    "incluir": 3,
    "falha": 1,
    "excluir": 1
  },
  "decomposicao": {
    "8B|incluir": {
      "dentro": 0.1007,
      "entre": 0.0936,
      "fracao_entre": 0.4816
    },
    "8B|falha": {
      "dentro": 0.0981,
      "entre": 0.0708,
      "fracao_entre": 0.4194
    },
    "8B|excluir": {
      "dentro": 0.0934,
      "entre": 0.0884,
      "fracao_entre": 0.4864
    },
    "14B|incluir": {
      "dentro": 0.1901,
      "entre": 0.046,
      "fracao_entre": 0.1947
    },
    "14B|falha": {
      "dentro": 0.1033,
      "entre": 0.0402,
      "fracao_entre": 0.28
    },
    "14B|excluir": {
      "dentro": 0.1094,
      "entre": 0.0577,
      "fracao_entre": 0.3455
    }
  }
}
